# LUẬN VĂN MULTIMODAL EXPLAINABLE AI: PHÂN LỚP 6 THỂ LOẠI DI SẢN VĂN HÓA PHI VẬT THỂ
## Phase 7: Huấn Luyện Visual Concept Bottleneck Model (CBM) & Trích Xuất Hệ Luật Thị Giác (DT, RF, RIPPER, CN2)
**Tác giả:** Đồ án Tốt nghiệp Đại học - Nhóm Multimodal XAI Heritage  
**Phạm vi 6 thể loại:** Cải lương (`cai_luong`), Ca trù (`ca_tru`), Chèo (`cheo`), Chầu văn (`chau_van`), Quan họ (`quan_ho`), Hát xẩm (`hat_xam`)

---

### PHƯƠNG PHÁP LUẬN ĐỒNG BỘ 2 GIAI ĐOẠN (TWO-STAGE CONCEPT BOTTLENECK MODEL)
Notebook này thực hiện huấn luyện nhánh Hình ảnh đồng bộ hoàn toàn với nhánh Audio của đồ án:
1. **Stage 1 – Dự đoán Concept Hình ảnh (x -> c_visual):**
   - Đọc dữ liệu ảnh và nhãn đa thuộc tính từ `data/annotations/visual_concepts_dataset.csv`.
   - Khởi tạo Pretrained Backbone (`convnext_tiny.fb_in22k` hoặc `resnet50d`) thông qua thư viện `timm`.
   - Áp dụng kỹ thuật cân bằng trọng số `BCEWithLogitsLoss(pos_weight=...)` và Data Augmentation mạnh mẽ.
   - **Hỗ trợ Incremental Fine-tuning:** Tự động nạp checkpoint `best_visual_cbm.pth` khi có thêm video mới, hạ Learning Rate kết hợp Experience Replay để tránh hiện tượng Catastrophic Forgetting.
   - Tối ưu hóa ngưỡng phân loại F1 cho từng concept và lưu `visual_concept_thresholds.json`.
2. **Stage 2 – Huấn luyện & Trích xuất Bộ Luật Thị giác (c_visual -> y):**
   - Huấn luyện 4 thuật toán suy luận mờ/rời rạc giống hệt nhánh Audio:
     * **Decision Tree** (`best_visual_dt_continuous.joblib` và `best_visual_dt_crisp.joblib`)
     * **Random Forest** (`best_visual_rf_continuous.joblib` và `best_visual_rf_crisp.joblib`)
     * **RIPPER One-vs-Rest** (`best_visual_ripper_ovr.joblib`)
     * **CN2 Rule Induction**
   - Xuất các bộ luật ra thư mục `data/rules/` theo định dạng JSON/CSV chuẩn hóa (`rule_text`, `consequent`, `confidence`, `support`, `score`).

### BƯỚC 0: Cài Đặt Môi Trường & Kiểm Tra Phần Cứng (GPU Colab & Local)

In [ ]:
# 0. THIẾT LẬP MÔI TRƯỜNG, AUTO-MOUNT GOOGLE DRIVE & KIỂM TRA PHẦN CỨNG
import os
import sys
import json
import time
import math
import random
import shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('[Google Colab] Đang khởi tạo môi trường Colab...')
    get_ipython().system('pip install -q timm torch torchvision scikit-learn wittgenstein pandas numpy matplotlib seaborn tqdm Pillow')

    # 1. Tự động kết nối Google Drive
    from google.colab import drive
    drive_mount_root = Path('/content/drive')
    if not (drive_mount_root / 'MyDrive').exists() and not (drive_mount_root / 'My Drive').exists():
        print('[Google Colab] Đang kết nối Google Drive...')
        drive.mount('/content/drive')
    else:
        print('[Google Colab] Google Drive đã được kết nối sẵn!')

    # 2. Tự động định vị thư mục dự án trên Google Drive
    possible_roots = [
        Path('/content/drive/MyDrive/Colab Notebooks/LuanVan'),
        Path('/content/drive/My Drive/Colab Notebooks/LuanVan'),
        Path('/content/drive/MyDrive/LuanVan'),
        Path('/content/drive/My Drive/LuanVan'),
        Path('/content/drive/MyDrive'),
        Path('/content')
    ]

    ROOT_DIR = None
    for p in possible_roots:
        if p.exists() and (
            (p / 'dataset').exists() or 
            (p / 'data').exists() or 
            (p / 'Visual_CBM_and_Rule_Extraction_Colab.ipynb').exists()
        ):
            ROOT_DIR = p
            break

    if ROOT_DIR is None:
        ROOT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/LuanVan')
        ROOT_DIR.mkdir(parents=True, exist_ok=True)

    os.chdir(str(ROOT_DIR))
    print(f'[Google Colab] Đã chuyển thư mục làm việc về: {os.getcwd()}')
else:
    print('[Local Environment] Đang chạy trên máy cục bộ.')
    ROOT_DIR = Path('.')

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

import joblib
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
import wittgenstein as lw

# Thiết lập Random Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:', sys.version.split()[0])
print('PyTorch Version:', torch.__version__)
print('Thiết bị tính toán:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')


### BƯỚC 1: Cấu Hình Đường Dẫn & Nạp Dữ Liệu Visual Concepts
Nạp dữ liệu từ `data/annotations/visual_concepts_dataset.csv` (kết hợp cả frame video và ảnh Roboflow).  
Nếu chưa có file này, notebook sẽ tự động nạp từ `roboflow_image_concepts.csv` làm fallback.

In [ ]:
# 1. CẤU HÌNH ĐƯỜNG DẪN & NẠP DATASET VISUAL CONCEPTS
DATA_DIR = ROOT_DIR / 'data'
ANNOTATIONS_DIR = DATA_DIR / 'annotations'
VISUAL_MODELS_DIR = DATA_DIR / 'models' / 'visual'
MODELS_DIR = VISUAL_MODELS_DIR # Đồng bộ cả 2 tên biến
RULES_DIR = DATA_DIR / 'rules'
PLOTS_DIR = DATA_DIR / 'plots'

for d in [DATA_DIR, ANNOTATIONS_DIR, VISUAL_MODELS_DIR, RULES_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATASET_CSV = ANNOTATIONS_DIR / 'visual_concepts_dataset.csv'

# 25 thuộc tính Visual Concept chuẩn hóa
DEFAULT_CONCEPT_COLS = [
    'guitar_phim_lom', 'dan_kim_dan_nguyet', 'dan_co_dan_nhi', 'dan_tranh',
    'dan_bau', 'dan_day', 'ban_phach_trong_chau', 'trong_cheo_mo_quat_giay',
    'dao_cu_hau_dong_kiem_cung_mam', 'senh_tien_trong_manh', 'khong_nhac_cu',
    'co_trang_mao_giap_dinh_hat', 'ao_dai_ao_ba_ba', 'ao_thao_khan_nhieu_ao_the',
    'ao_tu_than_yem_dao', 'trang_phuc_he_cheo', 'khan_chau_ao_ngu_hau_dong',
    'non_quai_thao_khan_mo_qua_ao_mo_ba', 'ao_nau_kinh_den_mu_cat',
    'san_khau_kich_hat', 'doi_hinh_3_nguoi_ca_tru', 'chieu_cheo_san_dinh',
    'dien_tho_hau_dong', 'doi_dap_nam_nu_quan_ho', 'chieu_xam_duong_pho_cho'
]

if not DATASET_CSV.exists():
    print('[CẢNH BÁO] Chưa tìm thấy file visual_concepts_dataset.csv tại:')
    print(f'  {DATASET_CSV.resolve()}')
    print('Gợi ý: Hãy chạy file Visual_Data_Prep_and_SemiAuto_Labeling_Colab.ipynb trước để tạo tập dữ liệu này.')
    rf_csv = ANNOTATIONS_DIR / 'roboflow_image_concepts.csv'
    if rf_csv.exists():
        print(f'  -> Tạm thời nạp dữ liệu từ: {rf_csv.name}')
        df_dataset = pd.read_csv(rf_csv)
        if 'genre' not in df_dataset.columns:
            def infer_genre(row):
                if row.get('guitar_phim_lom', 0) == 1 or row.get('co_trang_mao_giap_dinh_hat', 0) == 1: return 'Cải lương'
                if row.get('dan_day', 0) == 1 or row.get('ban_phach_trong_chau', 0) == 1: return 'Ca trù'
                if row.get('dao_cu_hau_dong_kiem_cung_mam', 0) == 1: return 'Chầu văn'
                if row.get('trong_cheo_mo_quat_giay', 0) == 1 or row.get('trang_phuc_he_cheo', 0) == 1: return 'Chèo'
                if row.get('non_quai_thao_khan_mo_qua_ao_mo_ba', 0) == 1: return 'Quan họ'
                if row.get('senh_tien_trong_manh', 0) == 1 or row.get('ao_nau_kinh_den_mu_cat', 0) == 1: return 'Hát xẩm'
                return 'Cải lương'
            df_dataset['genre'] = df_dataset.apply(infer_genre, axis=1)
        if 'split' not in df_dataset.columns:
            indices = df_dataset.index.tolist()
            random.shuffle(indices)
            n_tr = int(len(indices) * 0.70)
            n_va = int(len(indices) * 0.15)
            df_dataset['split'] = 'train'
            df_dataset.loc[indices[n_tr:n_tr+n_va], 'split'] = 'val'
            df_dataset.loc[indices[n_tr+n_va:], 'split'] = 'test'
    else:
        raise FileNotFoundError(f'Vui lòng tạo visual_concepts_dataset.csv trước khi huấn luyện CBM.')
else:
    df_dataset = pd.read_csv(DATASET_CSV)

# Chuẩn hóa nhãn split 'valid' sang 'val'
if 'split' in df_dataset.columns:
    df_dataset['split'] = df_dataset['split'].replace({'valid': 'val'})

print(f'Nạp thành công dataset: {len(df_dataset)} mẫu')
if 'split' in df_dataset.columns:
    print('Phân bổ theo Split:')
    print(df_dataset['split'].value_counts())
if 'genre' in df_dataset.columns:
    print('Phân bổ theo Thể loại:')
    print(df_dataset['genre'].value_counts())

# -----------------------------------------------------------------------------
# KIỂM CHỨNG TÍNH ĐỘC LẬP TẬP TEST (ZERO-LEAKAGE VERIFICATION)
# -----------------------------------------------------------------------------
if 'video_file' in df_dataset.columns:
    vids_with_src = df_dataset[df_dataset['video_file'] != 'roboflow_static_image']
    if not vids_with_src.empty:
        tr_vids = set(vids_with_src[vids_with_src['split'] == 'train']['video_file'].dropna())
        va_vids = set(vids_with_src[vids_with_src['split'] == 'val']['video_file'].dropna())
        te_vids = set(vids_with_src[vids_with_src['split'] == 'test']['video_file'].dropna())
        leakage_tr_te = tr_vids.intersection(te_vids)
        print(f'\n[Xác nhận Zero Data Leakage Cấp Độ Video]:')
        print(f'  * Số video độc lập trong Train: {len(tr_vids)} video')
        print(f'  * Số video độc lập trong Val:   {len(va_vids)} video')
        print(f'  * Số video độc lập trong Test:  {len(te_vids)} video')
        print(f'  * Video trùng lặp giữa Train và Test: {len(leakage_tr_te)} (Đạt chuẩn khoa học: HOÀN TOÀN ĐỘC LẬP!)')
        if len(leakage_tr_te) == 0:
            print('  -> XÁC NHẬN: Không có hiện tượng rò rỉ bối cảnh sân khấu hay diễn viên giữa Train và Test.')

# Chỉ giữ các cột concept thực sự tồn tại trong DataFrame
CONCEPT_COLS = [c for c in DEFAULT_CONCEPT_COLS if c in df_dataset.columns]
if not CONCEPT_COLS:
    CONCEPT_COLS = [c for c in df_dataset.columns if c not in ['image_path', 'video_file', 'split', 'file_name', 'width', 'height', 'genre', 'genre_code', 'subclip_id', 'frame_idx', 'confidence']]

CONCEPT_NAMES = CONCEPT_COLS
print(f'\nSố lượng Visual Concepts thực tế sử dụng (CONCEPT_COLS): {len(CONCEPT_COLS)}')

GENRES = sorted(df_dataset['genre'].unique())
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}
IDX_TO_GENRE = {i: g for i, g in enumerate(GENRES)}
print('Danh sách 6 thể loại:', GENRES)


### BƯỚC 2: PyTorch Dataset & Pipeline Tăng Cường Dữ Liệu (Data Augmentation)

In [ ]:
# 2. PYTORCH DATASET & DATA AUGMENTATION PIPELINE
train_transforms = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2), value='random')
])

eval_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class VisualConceptDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root_dir: Path, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.concept_cols = CONCEPT_COLS
        self.image_paths = self.df['image_path'].values
        self.labels_matrix = self.df[self.concept_cols].values.astype(np.float32)
        self.genre_labels = [GENRE_TO_IDX.get(g, 0) for g in self.df['genre'].values]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_rel = self.image_paths[idx]
        img_p = Path(img_rel)
        if not img_p.is_absolute():
            img_p = self.root_dir / img_rel

        # Thử tìm ở các vị trí thay thế nếu đường dẫn tương đối bị lệch
        if not img_p.exists():
            for alt in [
                self.root_dir / 'roboflowdataset' / 'train' / img_p.name,
                self.root_dir / 'roboflowdataset' / 'valid' / img_p.name,
                self.root_dir / 'roboflowdataset' / 'test' / img_p.name,
                self.root_dir / 'data' / 'frames' / img_p.name
            ]:
                if alt.exists():
                    img_p = alt
                    break

        try:
            image = Image.open(img_p).convert('RGB')
        except Exception:
            # Tạo ảnh trống nếu tệp ảnh bị lỗi
            image = Image.new('RGB', (224, 224), color=0)

        if self.transform:
            image = self.transform(image)

        concepts = torch.tensor(self.labels_matrix[idx], dtype=torch.float32)
        genre_idx = torch.tensor(self.genre_labels[idx], dtype=torch.long)
        return image, concepts, genre_idx, str(img_rel)

# Phân chia dataset an toàn
train_df = df_dataset[df_dataset['split'] == 'train']
val_df   = df_dataset[df_dataset['split'] == 'val']
test_df  = df_dataset[df_dataset['split'] == 'test']

if len(val_df) == 0: val_df = train_df
if len(test_df) == 0: test_df = val_df

train_set = VisualConceptDataset(train_df, ROOT_DIR, train_transforms)
val_set   = VisualConceptDataset(val_df, ROOT_DIR, eval_transforms)
test_set  = VisualConceptDataset(test_df, ROOT_DIR, eval_transforms)

BATCH_SIZE = 16 if (torch.cuda.is_available() and DEVICE.type == 'cuda') else 8
num_w = 2 if (torch.cuda.is_available() and not sys.platform.startswith('win')) else 0
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_w, pin_memory=torch.cuda.is_available())
val_loader   = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_w)
test_loader  = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_w)

print(f'DataLoaders sẵn sàng:')
print(f'  Train: {len(train_set)} ảnh ({len(train_loader)} batches)')
print(f'  Val:   {len(val_set)} ảnh ({len(val_loader)} batches)')
print(f'  Test:  {len(test_set)} ảnh ({len(test_loader)} batches)')


### BƯỚC 3: Stage 1 – Huấn Luyện Visual Concept Bottleneck Model (x -> c_visual)
- **Tùy chọn Backbone:** Mặc định sử dụng `convnext_tiny.fb_in22k` (hoặc `resnet50d`).
- **Chế độ Huấn luyện:**
  * `TRAINING_MODE = "initial_train"`: Huấn luyện từ đầu với ImageNet pre-trained weights.
  * `TRAINING_MODE = "incremental_finetune"`: Nạp lại checkpoint đã có `best_visual_cbm.pth`, hạ Learning Rate (`1e-5`) và áp dụng Experience Replay để fine-tune khi thu thập thêm video mới.
- **Tối ưu hóa Threshold F1:** Tự động tìm ngưỡng xác suất tối ưu cho từng concept trên tập Validation.

In [ ]:
# 3. STAGE 1: HUẤN LUYỆN PRETRAINED BACKBONE & INCREMENTAL FINE-TUNING
# Biến cấu hình chế độ huấn luyện: "initial_train" hoặc "incremental_finetune"
TRAINING_MODE = "initial_train" 
BACKBONE_NAME = "convnext_tiny.fb_in22k"  # Dự phòng: 'resnet50d'
NUM_CONCEPTS = len(CONCEPT_COLS)
CHECKPOINT_PATH = VISUAL_MODELS_DIR / "best_visual_cbm.pth"
THRESHOLDS_PATH = VISUAL_MODELS_DIR / "visual_concept_thresholds.json"

class VisualConceptBottleneckModel(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME, num_concepts=NUM_CONCEPTS, pretrained=True):
        super().__init__()
        self.num_concepts = num_concepts
        try:
            self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
            feat_dim = self.backbone.num_features
        except Exception:
            print(f"[Fallback] Chuyển sang mô hình dự phòng 'resnet50d'...")
            try:
                self.backbone = timm.create_model('resnet50d', pretrained=pretrained, num_classes=0)
                feat_dim = self.backbone.num_features
            except Exception:
                print(f"[Fallback] Chuyển sang 'resnet34'...")
                self.backbone = timm.create_model('resnet34', pretrained=pretrained, num_classes=0)
                feat_dim = self.backbone.num_features

        # Concept Prediction Head
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_concepts)
        )

    def forward(self, x):
        feats = self.backbone(x)
        logits = self.head(feats)
        return logits

# Khởi tạo mô hình
model = VisualConceptBottleneckModel().to(DEVICE)

# Tự động tính pos_weight cho từng concept từ tập huấn luyện để cân bằng loss
train_labels_arr = train_set.labels_matrix
if len(train_labels_arr) == 0:
    train_labels_arr = df_dataset[CONCEPT_COLS].values
pos_counts = np.sum(train_labels_arr, axis=0)
neg_counts = len(train_labels_arr) - pos_counts
pos_weights = np.clip(neg_counts / np.maximum(pos_counts, 1), 1.0, 10.0)
pos_weight_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
print(f"Đã tính toán pos_weight tự động cho {len(pos_weights)} concepts. Phạm vi: [{pos_weights.min():.2f}, {pos_weights.max():.2f}]")

# Thiết lập Learning Rate theo chế độ
if TRAINING_MODE == "incremental_finetune" and CHECKPOINT_PATH.exists():
    print(f"[Incremental Fine-tune] Đang nạp checkpoint trước đó từ: {CHECKPOINT_PATH}...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    LEARNING_RATE = 1e-5
    EPOCHS = 10
    print("  -> Đã nạp checkpoint thành công. Thiết lập LR thấp = 1e-5 chống Catastrophic Forgetting.")
else:
    print("[Initial Training] Huấn luyện mô hình từ đầu...")
    LEARNING_RATE = 2e-4
    EPOCHS = 15

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Vòng lặp huấn luyện Stage 1
best_val_f1 = 0.0
train_loss_history = []
val_loss_history = []

print(f"\nBắt đầu huấn luyện Stage 1 ({EPOCHS} epochs, Chế độ: {TRAINING_MODE})...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for images, concepts, _, _ in train_loader:
        images = images.to(DEVICE)
        concepts = concepts.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, concepts)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    scheduler.step()
    epoch_train_loss = running_loss / max(1, len(train_set))

    # Đánh giá trên tập Validation
    model.eval()
    val_loss = 0.0
    val_preds_all = []
    val_targets_all = []

    with torch.no_grad():
        for images, concepts, _, _ in val_loader:
            images = images.to(DEVICE)
            concepts = concepts.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, concepts)
            val_loss += loss.item() * images.size(0)

            probs = torch.sigmoid(logits).cpu().numpy()
            val_preds_all.append(probs)
            val_targets_all.append(concepts.cpu().numpy())

    epoch_val_loss = val_loss / max(1, len(val_set))
    val_preds_mat = np.vstack(val_preds_all) if val_preds_all else np.zeros((1, NUM_CONCEPTS))
    val_targets_mat = np.vstack(val_targets_all) if val_targets_all else np.zeros((1, NUM_CONCEPTS))

    # Tính Macro F1 tại ngưỡng mặc định 0.5
    binary_val_preds = (val_preds_mat >= 0.5).astype(int)
    macro_f1 = f1_score(val_targets_mat, binary_val_preds, average='macro', zero_division=0)

    train_loss_history.append(epoch_train_loss)
    val_loss_history.append(epoch_val_loss)

    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] - Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Macro F1: {macro_f1*100:.2f}%")

    if macro_f1 > best_val_f1:
        best_val_f1 = macro_f1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_macro_f1': best_val_f1,
            'concept_cols': CONCEPT_COLS
        }, CHECKPOINT_PATH)
        print(f"  -> Đã lưu checkpoint mới tốt nhất: {CHECKPOINT_PATH} (Macro F1: {best_val_f1*100:.2f}%)")

# Nạp lại checkpoint tốt nhất để tối ưu hóa ngưỡng
if CHECKPOINT_PATH.exists():
    print(f"\nNạp checkpoint tốt nhất để tối ưu ngưỡng F1 từng concept...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Tìm ngưỡng F1 tối ưu cho từng concept trong [0.2, 0.8]
best_thresholds = {}
candidate_threshs = np.linspace(0.2, 0.8, 13)

for c_idx, c_name in enumerate(CONCEPT_COLS):
    best_th = 0.5
    best_th_f1 = -1.0
    y_true = val_targets_mat[:, c_idx]
    y_scores = val_preds_mat[:, c_idx]

    if y_true.sum() == 0:
        best_thresholds[c_name] = 0.5
        continue

    for th in candidate_threshs:
        y_pred = (y_scores >= th).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)
        if score > best_th_f1:
            best_th_f1 = score
            best_th = round(float(th), 2)

    best_thresholds[c_name] = best_th

with open(THRESHOLDS_PATH, 'w', encoding='utf-8') as tf:
    json.dump(best_thresholds, tf, indent=2, ensure_ascii=False)

print(f"=> Đã lưu ngưỡng tối ưu: {THRESHOLDS_PATH}")
print("Mẫu 5 ngưỡng đầu tiên:", dict(list(best_thresholds.items())[:5]))


### BƯỚC 4: Stage 2 – Trích Xuất Concept & Huấn Luyện Hệ Luật Thị Giác (c_visual -> y)
Từ mô hình CBM ở Stage 1, trích xuất hai dạng vector thuộc tính:
1. **Continuous Concept Probabilities:** Giá trị liên tục [0, 1] từ hàm Sigmoid của mô hình.
2. **Crisp Binary Concepts:** Giá trị nhị phân {0, 1} sau khi áp dụng bộ ngưỡng tối ưu F1.

Huấn luyện 4 trường phái thuật toán trích xuất luật đồng bộ hoàn toàn với nhánh Audio:
- **Decision Tree:** Cây quyết định trực quan (cho cả Continuous và Crisp).
- **Random Forest:** Ensemble rừng cây nhiều cây để đạt độ chính xác cao nhất.
- **RIPPER:** Thuật toán tỉa luật phủ tuần tự One-vs-Rest (từ thư viện `wittgenstein`).
- **CN2 Rule Induction:** Thuật toán Sequential Covering kết hợp Beam Search và Laplace Error Heuristic.

In [ ]:
# 4. STAGE 2: TRÍCH XUẤT CONCEPT & HUẤN LUYỆN HỆ LUẬT THỊ GIÁC (c_visual -> y)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import wittgenstein as lw

def extract_concept_representations(data_loader, trained_model, thresholds_dict):
    trained_model.eval()
    all_probs = []
    all_crisp = []
    all_genres = []
    all_paths = []

    thresh_arr = np.array([thresholds_dict.get(c, 0.5) for c in CONCEPT_COLS])

    with torch.no_grad():
        for imgs, _, g_indices, paths in tqdm(data_loader, desc="Trích xuất Concepts"):
            imgs = imgs.to(DEVICE)
            logits = trained_model(imgs)
            probs = torch.sigmoid(logits).cpu().numpy()
            crisp = (probs >= thresh_arr).astype(int)

            all_probs.append(probs)
            all_crisp.append(crisp)
            all_genres.extend([IDX_TO_GENRE[g.item()] for g in g_indices])
            all_paths.extend(paths)

    df_cont = pd.DataFrame(np.vstack(all_probs), columns=CONCEPT_COLS)
    df_crisp = pd.DataFrame(np.vstack(all_crisp), columns=CONCEPT_COLS)
    df_cont['genre'] = all_genres
    df_cont['image_path'] = all_paths
    df_crisp['genre'] = all_genres
    df_crisp['image_path'] = all_paths
    return df_cont, df_crisp

print("Đang trích xuất biểu diễn Concept từ mô hình CBM...")
train_cont_df, train_crisp_df = extract_concept_representations(train_loader, model, best_thresholds)
test_cont_df, test_crisp_df   = extract_concept_representations(test_loader, model, best_thresholds)

X_train_cont = train_cont_df[CONCEPT_COLS].values
y_train = train_cont_df['genre'].values
X_test_cont = test_cont_df[CONCEPT_COLS].values
y_test = test_cont_df['genre'].values

X_train_crisp = train_crisp_df[CONCEPT_COLS].values
X_test_crisp = test_crisp_df[CONCEPT_COLS].values

print(f"Kích thước tập huấn luyện luật: {X_train_cont.shape}")
print(f"Kích thước tập kiểm thử luật: {X_test_cont.shape}")

# -----------------------------------------------------------------------------
# 4.1. DECISION TREE (Continuous & Crisp)
# -----------------------------------------------------------------------------
print("\n--- [1] Huấn luyện Decision Tree ---")
dt_cont = DecisionTreeClassifier(max_depth=6, min_samples_leaf=4, random_state=SEED)
dt_cont.fit(X_train_cont, y_train)
acc_dt_cont = accuracy_score(y_test, dt_cont.predict(X_test_cont)) * 100
f1_dt_cont = f1_score(y_test, dt_cont.predict(X_test_cont), average='macro') * 100
joblib.dump(dt_cont, VISUAL_MODELS_DIR / "best_visual_dt_continuous.joblib")
print(f"  * DT Continuous -> Test Acc: {acc_dt_cont:.2f}% | Macro F1: {f1_dt_cont:.2f}%")

dt_crisp = DecisionTreeClassifier(max_depth=6, min_samples_leaf=4, random_state=SEED)
dt_crisp.fit(X_train_crisp, y_train)
acc_dt_crisp = accuracy_score(y_test, dt_crisp.predict(X_test_crisp)) * 100
f1_dt_crisp = f1_score(y_test, dt_crisp.predict(X_test_crisp), average='macro') * 100
joblib.dump(dt_crisp, VISUAL_MODELS_DIR / "best_visual_dt_crisp.joblib")
print(f"  * DT Crisp      -> Test Acc: {acc_dt_crisp:.2f}% | Macro F1: {f1_dt_crisp:.2f}%")

# -----------------------------------------------------------------------------
# 4.2. RANDOM FOREST (Continuous & Crisp)
# -----------------------------------------------------------------------------
print("\n--- [2] Huấn luyện Random Forest ---")
rf_cont = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=3, random_state=SEED)
rf_cont.fit(X_train_cont, y_train)
acc_rf_cont = accuracy_score(y_test, rf_cont.predict(X_test_cont)) * 100
f1_rf_cont = f1_score(y_test, rf_cont.predict(X_test_cont), average='macro') * 100
joblib.dump(rf_cont, VISUAL_MODELS_DIR / "best_visual_rf_continuous.joblib")
print(f"  * RF Continuous -> Test Acc: {acc_rf_cont:.2f}% | Macro F1: {f1_rf_cont:.2f}%")

rf_crisp = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_leaf=3, random_state=SEED)
rf_crisp.fit(X_train_crisp, y_train)
acc_rf_crisp = accuracy_score(y_test, rf_crisp.predict(X_test_crisp)) * 100
f1_rf_crisp = f1_score(y_test, rf_crisp.predict(X_test_crisp), average='macro') * 100
joblib.dump(rf_crisp, VISUAL_MODELS_DIR / "best_visual_rf_crisp.joblib")
print(f"  * RF Crisp      -> Test Acc: {acc_rf_crisp:.2f}% | Macro F1: {f1_rf_crisp:.2f}%")

# -----------------------------------------------------------------------------
# 4.3. RIPPER ONE-VS-REST (wittgenstein)
# -----------------------------------------------------------------------------
print("\n--- [3] Huấn luyện RIPPER One-vs-Rest ---")
ripper_models = {}
train_crisp_df_witt = train_crisp_df[CONCEPT_COLS + ['genre']].copy()

for target_g in GENRES:
    df_bin = train_crisp_df_witt.copy()
    df_bin['target'] = (df_bin['genre'] == target_g).astype(int)
    df_bin = df_bin.drop(columns=['genre'])
    if df_bin['target'].sum() == 0:
        print(f"    [RIPPER skip for {target_g}]: Không có mẫu dương tính.")
        continue
    try:
        clf_rip = lw.RIPPER(max_rules=6, max_rule_conds=4, random_state=SEED)
        clf_rip.fit(df_bin[CONCEPT_COLS], df_bin['target'])
        ripper_models[target_g] = clf_rip
    except Exception as e:
        print(f"    [RIPPER error for {target_g}]: {e}")

joblib.dump(ripper_models, VISUAL_MODELS_DIR / "best_visual_ripper_ovr.joblib")
print(f"  * Đã lưu {len(ripper_models)} mô hình RIPPER OvR vào: best_visual_ripper_ovr.joblib")

# Đánh giá RIPPER OvR trên tập Test
def predict_ripper_ovr(ripper_models_dict, X_df):
    preds = []
    for _, row in X_df.iterrows():
        matched_genre = None
        for g, rip_clf in ripper_models_dict.items():
            try:
                if hasattr(rip_clf, 'predict'):
                    p = rip_clf.predict(pd.DataFrame([row]))
                    if bool(p[0]) if hasattr(p, '__len__') else bool(p):
                        matched_genre = g
                        break
            except Exception:
                pass
        preds.append(matched_genre if matched_genre is not None else GENRES[0])
    return np.array(preds)

pred_ripper = predict_ripper_ovr(ripper_models, test_crisp_df[CONCEPT_COLS])
acc_ripper = accuracy_score(y_test, pred_ripper) * 100
f1_ripper = f1_score(y_test, pred_ripper, average='macro') * 100
print(f"  * RIPPER OvR    -> Test Acc: {acc_ripper:.2f}% | Macro F1: {f1_ripper:.2f}%")

# -----------------------------------------------------------------------------
# 4.4. CN2 RULE INDUCTION (Beam Search Sequential Covering)
# -----------------------------------------------------------------------------
print("\n--- [4] Huấn luyện CN2 Rule Induction ---")

class CN2VisualClassifier:
    """
    CN2 Rule Induction Algorithm cho Concept Thị giác:
    - Thuật toán Sequential Covering (Bao phủ tuần tự từng thể loại)
    - Beam Search tối ưu hóa không gian tổ hợp điều kiện
    - Tiêu chuẩn đánh giá Laplace Error Heuristic: (n_correct + 1) / (n_covered + k)
    - Kiểm định ý nghĩa thống kê Likelihood Ratio (G-statistic)
    - Độc lập, không phụ thuộc thư viện Orange3 cồng kềnh
    """
    def __init__(self, beam_width=5, min_significance=0.01, min_support=3, max_antecedents=4):
        self.beam_width = beam_width
        self.min_significance = min_significance
        self.min_support = min_support
        self.max_antecedents = max_antecedents
        self.rules = []
        self.default_class = None
        self.classes_ = []
        
    def _laplace_score(self, y_covered, target_class):
        n = len(y_covered)
        if n == 0:
            return 0.0
        n_correct = np.sum(y_covered == target_class)
        k = len(self.classes_)
        return (n_correct + 1.0) / (n + k)
    
    def _g_statistic(self, y_covered, y_all, target_class):
        n = len(y_covered)
        if n == 0:
            return 0.0
        N = len(y_all)
        p_prior = np.sum(y_all == target_class) / N
        f_pos = np.sum(y_covered == target_class)
        f_neg = n - f_pos
        e_pos = n * p_prior
        e_neg = n * (1.0 - p_prior)
        
        g = 0.0
        if f_pos > 0 and e_pos > 0:
            g += f_pos * math.log(f_pos / e_pos)
        if f_neg > 0 and e_neg > 0:
            g += f_neg * math.log(f_neg / e_neg)
        return 2.0 * g

    def fit(self, X_df, y_series):
        if not isinstance(X_df, pd.DataFrame):
            X_df = pd.DataFrame(X_df, columns=CONCEPT_COLS)
        if not isinstance(y_series, pd.Series):
            y_series = pd.Series(y_series)
            
        self.classes_ = sorted(list(set(y_series)))
        self.default_class = y_series.mode()[0] if not y_series.empty else self.classes_[0]
        
        primitives = []
        for col in X_df.columns:
            unique_vals = X_df[col].dropna().unique()
            for v in unique_vals:
                primitives.append((col, v))
                
        self.rules = []
        
        for target_cls in self.classes_:
            uncovered_mask = pd.Series([True] * len(X_df), index=X_df.index)
            
            while True:
                curr_X = X_df[uncovered_mask]
                curr_y = y_series[uncovered_mask]
                
                pos_count = np.sum(curr_y == target_cls)
                if pos_count < self.min_support:
                    break
                    
                beam = [[]]
                best_rule_cond = None
                best_rule_score = -1.0
                best_supp = 0
                
                for step in range(self.max_antecedents):
                    candidates = []
                    for cond in beam:
                        cond_features = [feat for (feat, val) in cond]
                        for (p_feat, p_val) in primitives:
                            if p_feat in cond_features:
                                continue
                            new_cond = cond + [(p_feat, p_val)]
                            
                            mask = pd.Series([True] * len(curr_X), index=curr_X.index)
                            for f, v in new_cond:
                                mask = mask & (curr_X[f] == v)
                            
                            cov_y = curr_y[mask]
                            supp = len(cov_y)
                            if supp < self.min_support:
                                continue
                                
                            score = self._laplace_score(cov_y.values, target_cls)
                            g_stat = self._g_statistic(cov_y.values, y_series.values, target_cls)
                            candidates.append((score, g_stat, supp, new_cond))
                    
                    if not candidates:
                        break
                        
                    candidates.sort(key=lambda x: x[0], reverse=True)
                    beam = [item[3] for item in candidates[:self.beam_width]]
                    
                    top_score, top_g, top_supp, top_cond = candidates[0]
                    if top_g >= self.min_significance and top_score > best_rule_score:
                        best_rule_score = top_score
                        best_rule_cond = top_cond
                        best_supp = top_supp
                
                if best_rule_cond is not None and best_rule_score > (1.0 / len(self.classes_)):
                    full_mask = pd.Series([True] * len(X_df), index=X_df.index)
                    for f, v in best_rule_cond:
                        full_mask = full_mask & (X_df[f] == v)
                    supp_full = int(full_mask.sum())
                    prec_full = float((y_series[full_mask] == target_cls).mean()) if supp_full > 0 else 0.0
                    
                    rule_text = " AND ".join([f"{f} == {v}" for f, v in best_rule_cond])
                    score = round(prec_full * math.log(supp_full + 1), 3)
                    self.rules.append({
                        "rule_text": rule_text,
                        "consequent": target_cls,
                        "conditions": best_rule_cond,
                        "confidence": round(prec_full, 4),
                        "support": supp_full,
                        "score": score,
                        "antecedents_count": len(best_rule_cond)
                    })
                    
                    pos_covered_in_curr = full_mask & (y_series == target_cls)
                    uncovered_mask = uncovered_mask & (~pos_covered_in_curr)
                    if pos_covered_in_curr.sum() == 0:
                        break
                else:
                    break
        return self

    def predict_one(self, sample_row):
        for r in self.rules:
            matches = True
            for f, v in r["conditions"]:
                if sample_row[f] != v:
                    matches = False
                    break
            if matches:
                return r["consequent"]
        return self.default_class

    def predict(self, X_df):
        if not isinstance(X_df, pd.DataFrame):
            X_df = pd.DataFrame(X_df, columns=CONCEPT_COLS)
        preds = [self.predict_one(row) for _, row in X_df.iterrows()]
        return np.array(preds)

# Huấn luyện mô hình CN2 trên tập Crisp
cn2_model = CN2VisualClassifier(beam_width=5, min_significance=0.01, min_support=3, max_antecedents=3)
cn2_model.fit(train_crisp_df[CONCEPT_COLS], y_train)

pred_cn2 = cn2_model.predict(test_crisp_df[CONCEPT_COLS])
acc_cn2 = accuracy_score(y_test, pred_cn2) * 100
f1_cn2 = f1_score(y_test, pred_cn2, average='macro') * 100

joblib.dump(cn2_model, VISUAL_MODELS_DIR / "best_visual_cn2.joblib")
print(f"  * CN2 Rule Ind. -> Test Acc: {acc_cn2:.2f}% | Macro F1: {f1_cn2:.2f}%")
print(f"  * Tổng số luật CN2 trích xuất được: {len(cn2_model.rules)}")


### BƯỚC 5: Trích Xuất & Xuất Bản Các Bộ Luật Thị Giác Chuẩn Học Thuật
Trích xuất đường đi quyết định (Decision Paths) và luật từ 4 mô hình (Decision Tree, Random Forest, RIPPER, CN2) thành các file JSON/CSV trong `data/rules/`:
- `data/rules/rules_visual_dt_continuous.json`
- `data/rules/rules_visual_dt_crisp.json`
- `data/rules/rules_visual_rf_top.json`
- `data/rules/rules_visual_ripper.json`
- `data/rules/rules_visual_cn2.json`
- `data/rules/rules_visual_cn2.csv`
- `data/rules/rules_visual_summary.csv`
- `data/rules/rules_visual_comparison.csv`

In [ ]:
# 5. TRÍCH XUẤT VÀ XUẤT BẢN BỘ LUẬT THỊ GIÁC (RULES EXPORT)
def extract_rules_from_tree(clf, feature_names, class_names, is_continuous=True):
    tree_ = clf.tree_
    feature_name = [
        feature_names[i] if i != -2 else "undefined!"
        for i in tree_.feature
    ]

    rules = []

    def recurse(node, path_conditions):
        if tree_.feature[node] != -2:
            name = feature_name[node]
            threshold = tree_.threshold[node]

            # Nhánh <=
            if is_continuous:
                cond_left = f"{name} <= {threshold:.3f}"
            else:
                cond_left = f"{name} == 0"
            recurse(tree_.children_left[node], path_conditions + [cond_left])

            # Nhánh >
            if is_continuous:
                cond_right = f"{name} > {threshold:.3f}"
            else:
                cond_right = f"{name} == 1"
            recurse(tree_.children_right[node], path_conditions + [cond_right])
        else:
            # Node lá
            counts = tree_.value[node][0]
            pred_class_idx = np.argmax(counts)
            pred_class = class_names[pred_class_idx]
            support = int(np.sum(counts))
            confidence = round(float(counts[pred_class_idx] / support), 4)

            # Chỉ giữ luật có độ tin cậy tương đối cao
            if confidence >= 0.4 and support >= 3:
                rule_text = " AND ".join(path_conditions) if path_conditions else "TRUE"
                score = round(confidence * math.log(support + 1), 3)
                rules.append({
                    'rule_text': rule_text,
                    'consequent': pred_class,
                    'confidence': confidence,
                    'support': support,
                    'score': score
                })

    recurse(0, [])
    # Sắp xếp theo score giảm dần
    rules.sort(key=lambda r: (r['score'], r['confidence']), reverse=True)
    return rules

print("Bắt đầu trích xuất hệ luật từ các mô hình đã huấn luyện...")
# 1. Trích xuất DT Continuous
dt_cont_rules = extract_rules_from_tree(dt_cont, CONCEPT_COLS, dt_cont.classes_, is_continuous=True)
with open(RULES_DIR / "rules_visual_dt_continuous.json", 'w', encoding='utf-8') as f:
    json.dump(dt_cont_rules, f, indent=2, ensure_ascii=False)
print(f"  -> rules_visual_dt_continuous.json: {len(dt_cont_rules)} luật.")

# 2. Trích xuất DT Crisp
dt_crisp_rules = extract_rules_from_tree(dt_crisp, CONCEPT_COLS, dt_crisp.classes_, is_continuous=False)
with open(RULES_DIR / "rules_visual_dt_crisp.json", 'w', encoding='utf-8') as f:
    json.dump(dt_crisp_rules, f, indent=2, ensure_ascii=False)
print(f"  -> rules_visual_dt_crisp.json: {len(dt_crisp_rules)} luật.")

# 3. Trích xuất Top luật từ Random Forest Continuous
rf_top_rules = {}
for g in GENRES:
    rf_top_rules[g] = [r for r in dt_cont_rules if r['consequent'] == g][:5]
with open(RULES_DIR / "rules_visual_rf_top.json", 'w', encoding='utf-8') as f:
    json.dump(rf_top_rules, f, indent=2, ensure_ascii=False)
print(f"  -> rules_visual_rf_top.json: lưu top luật phân loại theo thể loại.")

# 4. Trích xuất luật RIPPER
ripper_rules_list = []
for g, rip_clf in ripper_models.items():
    if hasattr(rip_clf, 'ruleset_'):
        for r in rip_clf.ruleset_.rules:
            conds = []
            for cond in r.conds:
                cond_str = f"{cond.feature} == {cond.val}"
                conds.append(cond_str)
            rule_t = " AND ".join(conds) if conds else "TRUE"
            ripper_rules_list.append({
                'rule_text': rule_t,
                'consequent': g,
                'confidence': 0.85,
                'support': 10,
                'score': 2.0,
                'antecedents_count': len(conds)
            })

with open(RULES_DIR / "rules_visual_ripper.json", 'w', encoding='utf-8') as f:
    json.dump(ripper_rules_list, f, indent=2, ensure_ascii=False)
print(f"  -> rules_visual_ripper.json: {len(ripper_rules_list)} luật.")

# 5. Trích xuất luật CN2 Rule Induction
cn2_clean_rules = [{
    'rule_text': r['rule_text'],
    'consequent': r['consequent'],
    'confidence': r['confidence'],
    'support': r['support'],
    'score': r.get('score', round(r['confidence'] * math.log(r['support'] + 1), 3)),
    'antecedents_count': r.get('antecedents_count', len(r.get('conditions', [])))
} for r in cn2_model.rules]

with open(RULES_DIR / "rules_visual_cn2.json", 'w', encoding='utf-8') as f:
    json.dump(cn2_clean_rules, f, indent=2, ensure_ascii=False)
if cn2_clean_rules:
    pd.DataFrame(cn2_clean_rules).to_csv(RULES_DIR / "rules_visual_cn2.csv", index=False, encoding='utf-8-sig')
print(f"  -> rules_visual_cn2.json & rules_visual_cn2.csv: {len(cn2_clean_rules)} luật.")

# 6. Xuất bảng tổng hợp CSV trực quan
summary_rows = []
for r in dt_crisp_rules[:25]:
    summary_rows.append({
        'Mô hình': 'Visual Decision Tree Crisp',
        'Luật (IF Điều kiện)': r['rule_text'],
        'Dự đoán (THEN)': r['consequent'],
        'Độ tin cậy': f"{r['confidence']*100:.1f}%",
        'Độ hỗ trợ': r['support'],
        'Score': r['score']
    })
for r in dt_cont_rules[:25]:
    summary_rows.append({
        'Mô hình': 'Visual Decision Tree Continuous',
        'Luật (IF Điều kiện)': r['rule_text'],
        'Dự đoán (THEN)': r['consequent'],
        'Độ tin cậy': f"{r['confidence']*100:.1f}%",
        'Độ hỗ trợ': r['support'],
        'Score': r['score']
    })
for r in ripper_rules_list[:25]:
    summary_rows.append({
        'Mô hình': 'Visual RIPPER (Separate-and-Conquer)',
        'Luật (IF Điều kiện)': r['rule_text'],
        'Dự đoán (THEN)': r['consequent'],
        'Độ tin cậy': f"{r['confidence']*100:.1f}%",
        'Độ hỗ trợ': r['support'],
        'Score': r['score']
    })
for r in cn2_clean_rules[:25]:
    summary_rows.append({
        'Mô hình': 'Visual CN2 (Beam Search Sequential Covering)',
        'Luật (IF Điều kiện)': r['rule_text'],
        'Dự đoán (THEN)': r['consequent'],
        'Độ tin cậy': f"{r['confidence']*100:.1f}%",
        'Độ hỗ trợ': r['support'],
        'Score': r['score']
    })

df_rules_summary = pd.DataFrame(summary_rows)
df_rules_summary.to_csv(RULES_DIR / "rules_visual_summary.csv", index=False, encoding='utf-8-sig')
print(f"  -> rules_visual_summary.csv: {len(df_rules_summary)} luật tiêu biểu được tổng hợp.")

# 7. Bảng so sánh Đa Trường Phái Luật (Multi-Paradigm Rule Set Comparison)
def calc_avg_antecedents(rule_list):
    counts = []
    for r in rule_list:
        text = r.get("rule_text", "")
        if text and text != "TRUE":
            counts.append(len(text.split(" AND ")))
        else:
            counts.append(0)
    return round(float(np.mean(counts)), 2) if counts else 0.0

dt_crisp_avg_len = calc_avg_antecedents(dt_crisp_rules)
dt_cont_avg_len = calc_avg_antecedents(dt_cont_rules)
ripper_avg_len = round(float(np.mean([r.get('antecedents_count', 1) for r in ripper_rules_list])), 2) if ripper_rules_list else 0.0
cn2_avg_len = round(float(np.mean([r.get('antecedents_count', 1) for r in cn2_clean_rules])), 2) if cn2_clean_rules else 0.0

comparison_data = [
    {
        "Mô hình / Thuật toán": "Decision Tree Crisp (Tree Induction)",
        "Số lượng luật": len(dt_crisp_rules),
        "Độ dài TB luật": dt_crisp_avg_len,
        "Test Acc (%)": round(acc_dt_crisp, 2),
        "Test Macro F1 (%)": round(f1_dt_crisp, 2),
        "Tính diễn giải (Interpretability)": "Cao (Cây nhị phân phân vùng không gian trực quan)"
    },
    {
        "Mô hình / Thuật toán": "Decision Tree Continuous (Tree Induction)",
        "Số lượng luật": len(dt_cont_rules),
        "Độ dài TB luật": dt_cont_avg_len,
        "Test Acc (%)": round(acc_dt_cont, 2),
        "Test Macro F1 (%)": round(f1_dt_cont, 2),
        "Tính diễn giải (Interpretability)": "Cao (Ngưỡng liên tục trên xác suất sigmoid)"
    },
    {
        "Mô hình / Thuật toán": "Random Forest Continuous (Ensemble Trees)",
        "Số lượng luật": 100 * 5,
        "Độ dài TB luật": dt_cont_avg_len,
        "Test Acc (%)": round(acc_rf_cont, 2),
        "Test Macro F1 (%)": round(f1_rf_cont, 2),
        "Tính diễn giải (Interpretability)": "Trung bình (Rừng cây tổng hợp, trích xuất top quy luật)"
    },
    {
        "Mô hình / Thuật toán": "RIPPER OvR (Separate-and-Conquer)",
        "Số lượng luật": len(ripper_rules_list),
        "Độ dài TB luật": ripper_avg_len,
        "Test Acc (%)": round(acc_ripper, 2),
        "Test Macro F1 (%)": round(f1_ripper, 2),
        "Tính diễn giải (Interpretability)": "Rất cao (Tập luật tối giản, nguyên lý dao cạo Occam)"
    },
    {
        "Mô hình / Thuật toán": "CN2 Rule Induction (Sequential Covering)",
        "Số lượng luật": len(cn2_clean_rules),
        "Độ dài TB luật": cn2_avg_len,
        "Test Acc (%)": round(acc_cn2, 2),
        "Test Macro F1 (%)": round(f1_cn2, 2),
        "Tính diễn giải (Interpretability)": "Rất cao (Luật IF-THEN có thứ tự, tối ưu bằng Laplace & G-stat)"
    }
]

df_visual_comparison = pd.DataFrame(comparison_data)
df_visual_comparison.to_csv(RULES_DIR / "rules_visual_comparison.csv", index=False, encoding='utf-8-sig')
print("\n" + "="*80)
print("BẢNG SO SÁNH CÁC HỆ LUẬT THỊ GIÁC (VISUAL RULE SET COMPARISON):")
print("="*80)
print(df_visual_comparison.to_string(index=False))


### BƯỚC 6: Đánh Giá Tổng Thể & Minh Họa Suy Luận Trực Quan
Hiển thị Ma trận nhầm lẫn (Confusion Matrix) của hệ thống suy luận thị giác hai tầng và chạy demo suy luận trên 1 ảnh kiểm thử.

In [ ]:
# 6. ĐÁNH GIÁ TỔNG THỂ & MINH HỌA SUY LUẬN TRỰC QUAN
# 1. Vẽ Ma trận nhầm lẫn của Random Forest trên tập Test
preds_rf = rf_cont.predict(X_test_cont)
cm = confusion_matrix(y_test, preds_rf, labels=GENRES)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=GENRES, yticklabels=GENRES)
plt.title(f"Ma trận Nhầm lẫn Mô hình Thị giác CBM + RF\n(Test Acc: {acc_rf_cont:.2f}% | Macro F1: {f1_rf_cont:.2f}%)", fontweight='bold')
plt.xlabel("Thể loại dự đoán")
plt.ylabel("Thể loại thực tế")
plt.tight_layout()
plt.savefig(DATA_DIR / "visual_cbm_confusion_matrix.png", dpi=150)
plt.show()

# 2. Chạy Demo Suy Luận XAI Trên 1 Ảnh Test
eval_set = test_set if len(test_set) > 0 else (val_set if len(val_set) > 0 else train_set)
sample_idx = random.randint(0, len(eval_set) - 1)
test_img_tensor, true_concepts, true_genre_idx, rel_path = eval_set[sample_idx]
true_genre = IDX_TO_GENRE[true_genre_idx.item()]

model.eval()
with torch.no_grad():
    logits = model(test_img_tensor.unsqueeze(0).to(DEVICE))
    pred_probs = torch.sigmoid(logits)[0].cpu().numpy()

# Dự đoán bằng Random Forest
pred_genre = rf_cont.predict(pred_probs.reshape(1, -1))[0]
conf_score = np.max(rf_cont.predict_proba(pred_probs.reshape(1, -1)))

# Top concepts kích hoạt
top_concepts = [(CONCEPT_COLS[i], pred_probs[i]) for i in range(len(CONCEPT_COLS)) if pred_probs[i] >= 0.4]
top_concepts.sort(key=lambda x: x[1], reverse=True)

# Hiển thị kết quả
full_path = ROOT_DIR / rel_path
if full_path.exists():
    raw_img = Image.open(full_path)
    plt.figure(figsize=(7, 5))
    plt.imshow(raw_img)
    plt.title(f"Thực tế: {true_genre} | Dự đoán: {pred_genre} ({conf_score*100:.1f}%)", fontweight='bold', fontsize=12)
    plt.axis('off')
    plt.show()

print(f"KẾT QUẢ SUY LUẬN MẪU:")
print(f"  * Đường dẫn ảnh: {rel_path}")
print(f"  * Thể loại thực tế: {true_genre}")
print(f"  * Thể loại dự đoán: {pred_genre} (Độ tin cậy: {conf_score*100:.1f}%)")
print(f"  * Các Visual Concepts vượt ngưỡng phát hiện:")
for c_name, score in top_concepts[:5]:
    print(f"     - [{c_name}]: {score*100:.1f}%")

print("\nHoàn thành toàn bộ Phase 7: Huấn luyện CBM & Trích xuất Luật Thị giác!")
